In [1]:
from lmms_eval.models.qwen2_5_vl import Qwen2_5_VL
from data.utils_strokerehab import DataPaths
from lmms_eval.models.model_utils.load_video import load_long_video_decord

/gpfs/scratch/vhl2022/miniconda3/envs/cvfm4rehab/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def to_single_line_escaped(multiline_str: str) -> str:
    """
    Convert a multi-line string into a single-line string
    with escaped double quotes.

    Example:
        input:
            {
              "question": "How well does the patient perform forearm supination?",
              "scoring_criteria": {"0": "Cannot be performed."}
            }

        output:
            "{""question"": ""How well does the patient perform forearm supination?"", ""scoring_criteria"": {""0"": ""Cannot be performed.""}}"
    """
    # Strip leading/trailing whitespace and newlines
    cleaned = multiline_str.strip()
    # Collapse to one line
    single_line = " ".join(cleaned.split())
    # Escape double quotes by doubling them
    escaped = single_line.replace('"', '""')
    # Wrap in quotes
    return f'"{escaped}"'

In [3]:
model = Qwen2_5_VL(pretrained="Qwen/Qwen2.5-VL-7B-Instruct")

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
Loading checkpoint shards: 100%|██████████| 5/5 [00:16<00:00,  3.37s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


In [4]:
from qwen_vl_utils import process_vision_info

def build_input_ids(context, pil_frames):
    messages = []
    msg = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",
            "content": [
            {"type": "video", "video": pil_frames, "max_pixels": model.max_pixels, "fps": model.sampling_fps},
            {"type": "text", "text": context} 
            ]},
    ]
    messages.append(msg)
    text = model.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs, video_kwargs = process_vision_info(messages, return_video_kwargs=True)
    inputs = model.processor(
        text=text,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
        **video_kwargs
    )
    if model.device_map == "auto":
        inputs = inputs.to("cuda")
    else:
        inputs = inputs.to(model.device)
    return inputs

In [5]:
from lmms_eval.tasks.strokerehab.utils_impairment_analysis import load_strokerehab_ia_dataset

ds = load_strokerehab_ia_dataset(patients='all', fm_items='3-8', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']

In [6]:
import os
import numpy as np
from PIL import Image
import torch

def get_output(path_v, context):
    video_path = os.path.join(DataPaths.IA_CLIPPED_VIDEO_DIR, path_v)
    videos = load_long_video_decord(video_path, 32, sampling_strategy='uniform', overlap_frames_num=-1, sampling_fps=-1)
    frames, start_time, end_time = next(videos)
    def build_pil_frames(video):
        return [Image.fromarray(frame.astype(np.uint8)) for frame in video]
    pil_frames = build_pil_frames(frames)
    input_ids = build_input_ids(context, pil_frames)

    with torch.inference_mode():
        cont = model.model.generate(
            **input_ids,
            eos_token_id=model.tokenizer.eos_token_id,
            pad_token_id=model.tokenizer.pad_token_id,
            do_sample=False,
            # temperature=1e-5,
            top_p=1.0,
            num_beams=1,
            max_new_tokens=256,
        )

    generated_ids_trimmed = [out_ids[len(in_ids) :] for in_ids, out_ids in zip(input_ids.input_ids, cont)]
    text_outputs = model.processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
    return text_outputs

In [66]:
# 3
context_3 = \
"""
{
  "question": "How well does the patient raise their shoulder (shoulder elevation)?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "instructions": "Task cue: 'Starting with your arm in your lap, bring your hand to your ear like you're answering the telephone. Get your elbow up to shoulder height, get your shoulder back, and flip your hand so your thumb is pointed toward the ceiling.' For this item, look only at the final position. Check if the shoulder itself is raised higher than the starting neutral position. Do not count extra leaning of the body or tilting of the head as correct. Base the score only on how the shoulder ends up.",
  "scoring_criteria": {
    "0": "Cannot be performed. No shoulder lift at all, or needs help, or uses a lot of body/head lean instead.",
    "1": "Performed partially. Shoulder lifts somewhat but not all the way, or depends a lot on leaning the body/head.",
    "2": "Performed faultlessly. Shoulder is clearly lifted and held at the end (e.g. the patient achieves the telephone-answering position), with little or no body/head leaning. "
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_3))

Unused or unrecognized kwargs: return_tensors, fps.


C00011/C00011_FM3_8_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient raises their shoulder to approximately shoulder height, which is consistent with the instructions given. The shoulder appears to be elevated without significant body or head leaning, indicating that the patient has achieved the desired position.

FINAL_SCORE: 2
S0001/S0001_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient raises their shoulder to approximately shoulder height, with the elbow reaching close to the ear. The shoulder appears to be elevated without significant body or head leaning. The patient's hand is positioned near the ear, indicating that the shoulder has been lifted correctly.

FINAL_SCORE: 2
S00021/S00021_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient raises their shoulder slightly, but it does not reach shoulder height. The shoulder lift is minimal and appears to rely more on body and head movement rather than a clear shoulder elevation.

FINAL_SCORE: 0
S0005/S0005_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient raises their shoulder slightly, but it does not reach shoulder height. The shoulder lift is partial, and there is noticeable body leaning to achieve the position. 

FINAL_SCORE: 1


In [67]:
# 4
context_4 = \
"""
{
  "question": "How well does the patient retract their shoulder?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "instructions": "Task cue: 'Starting with your arm in your lap, bring your hand to your ear like you're answering the telephone. Get your elbow up to shoulder height, get your shoulder back, and flip your hand so your thumb is pointed toward the ceiling.' For this item, look only at the final position. Check how well the patient pulls their shoulder backwards. Base the score only on how the shoulder ends up.",
  "scoring_criteria": {
    "0": "Cannot be performed. No shoulder retraction at all, or retraction does not make it to behind the back.",
    "1": "Performed partially. Shoulder retracts somewhat but not all the way, or depends a lot on leaning the body/head.",
    "2": "Performed faultlessly. Shoulder is clearly retracted and held at the end (e.g. the patient achieves the telephone-answering position), with little or no body/head leaning."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_4))

C00011/C00011_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Unused or unrecognized kwargs: return_tensors, fps.


The patient's shoulder retracts to a position where it is clearly behind the back, achieving the telephone-answering position without any significant body or head leaning.

FINAL_SCORE: 2
S0001/S0001_FM3_8_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient brings their hand to their ear as if answering the telephone, raises their elbow to shoulder height, and then retracts their shoulder backward. The shoulder appears to be retracted to a point where it is clearly behind the back, without any significant body or head leaning.

FINAL_SCORE: 2
S00021/S00021_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's shoulder is retracted and held at the end of the movement, with minimal body or head leaning. The shoulder appears to be clearly retracted and positioned as if answering the telephone.

FINAL_SCORE: 2
S0005/S0005_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's shoulder is retracted to a point where it is clearly behind the back, and there is minimal body or head leaning. The shoulder appears to be held in place without much assistance.

FINAL_SCORE: 2


In [68]:
# 5
context_5 = \
"""
{
  "question": "How well does the patient abduct their shoulder (arm out to side >= 90 degrees)?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "instructions": "Task cue: 'Starting with your arm in your lap, bring your hand to your ear like you're answering the telephone. Get your elbow up to shoulder height, get your shoulder back, and flip your hand so your thumb is pointed toward the ceiling.' For this item, look only at the final position. Check how well the patient gets their arm out to the side (elbow at shoulder's height level).",
  "scoring_criteria": {
    "0": "Cannot be performed. Arm cannot be lifted out to the side.",
    "1": "Performed partially. Arm is lifted somewhat to the side, but not all the way.",
    "2": "Performed faultlessly. Arm is lifted all the way out to the side, to around shoulder height."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_5))

C00011/C00011_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Unused or unrecognized kwargs: return_tensors, fps.


The patient successfully brings her arm out to the side, reaching approximately shoulder height. She can lift her elbow up to shoulder height and get her shoulder back, with her hand positioned as if answering the telephone. The patient's arm is extended out to the side, demonstrating good abduction.

FINAL_SCORE: 2
S0001/S0001_FM3_8_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient brings their arm up to shoulder height and then flips their hand so that the thumb points towards the ceiling. The arm is lifted out to the side, reaching approximately shoulder height.

FINAL_SCORE: 2
S00021/S00021_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to have difficulty lifting their arm out to the side. The arm is raised slightly but does not reach shoulder height or beyond. The shoulder abduction is limited.

FINAL_SCORE: 0
S0005/S0005_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient brings their arm up to shoulder height and then flips their hand so that the thumb points towards the ceiling. The arm is lifted out to the side, reaching approximately shoulder height.

FINAL_SCORE: 2


In [69]:
# 6
context_6 = \
"""
{
  "question": "How well does the patient perform external rotation (elbow crease fully visible)?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "instructions": "Task cue: 'Starting with your arm in your lap, bring your hand to your ear like you're answering the telephone. Get your elbow up to shoulder height, get your shoulder back, and flip your hand so your thumb is pointed toward the ceiling.' For this item, look only at the final position. Check how well the patient gets their arm into external rotation (elbow crease fully visible).",
  "scoring_criteria": {
    "0": "Cannot be performed. No outward rotation occurs.",
    "1": "Performed partially. The forearm moves outward a little but not fully.",
    "2": "Performed faultlessly. Elbow crease is fully visible."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_6))

Unused or unrecognized kwargs: return_tensors, fps.


C00011/C00011_FM3_8_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Unused or unrecognized kwargs: return_tensors, fps.


The patient brings her hand to her ear as if answering the telephone, raises her elbow to shoulder height, and then flips her hand so that her thumb points toward the ceiling. Her elbow crease is fully visible during this movement.

FINAL_SCORE: 2
S0001/S0001_FM3_8_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient brings his hand to his ear as if answering the telephone, raises his elbow to shoulder height, and then flips his hand so that his thumb points toward the ceiling. The elbow crease is fully visible during this movement.

FINAL_SCORE: 2
S00021/S00021_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to have difficulty bringing the elbow up to shoulder height and flipping the hand so that the thumb points towards the ceiling. The elbow crease is not fully visible during the movement.

FINAL_SCORE: 0
S0005/S0005_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to have difficulty bringing the elbow up to shoulder height and flipping the hand so that the thumb points towards the ceiling. The elbow crease is not fully visible during the movement.

FINAL_SCORE: 0


In [70]:
# 7
context_7 = \
"""
{
  "question": "How well does the patient perform elbow flexion?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "instructions": "Task cue: 'Starting with your arm in your lap, bring your hand to your ear like you're answering the telephone. Get your elbow up to shoulder height, get your shoulder back, and flip your hand so your thumb is pointed toward the ceiling.' For this item, look only at the final position. Check how well the patient bends their elbow.",
  "scoring_criteria": {
    "0": "Cannot be performed. No elbow flexion occurs.",
    "1": "Performed partially. The elbow bends somewhat but the range is limited.",
    "2": "Performed faultlessly. The elbow can achieve a typical range of motion."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_7))

Unused or unrecognized kwargs: return_tensors, fps.


C00011/C00011_FM3_8_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Unused or unrecognized kwargs: return_tensors, fps.


The patient brings her hand to her ear as if answering the telephone, then raises her elbow to shoulder height. She successfully flips her hand so that her thumb points towards the ceiling. Her elbow reaches close to shoulder height, indicating good range of motion.

FINAL_SCORE: 2
S0001/S0001_FM3_8_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient brings his hand to his ear as if answering the telephone, raises his elbow to shoulder height, and then flips his hand so that his thumb points towards the ceiling. However, the elbow does not fully reach shoulder height, indicating a limitation in the range of motion.

FINAL_SCORE: 1
S00021/S00021_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to have difficulty bringing their hand to their ear and bending their elbow to shoulder height. The elbow does not reach the shoulder level, indicating limited range of motion.

FINAL_SCORE: 0
S0005/S0005_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient brings his hand to his ear as if answering the telephone, raises his elbow to shoulder height, and then flips his hand so that his thumb points toward the ceiling. However, the elbow does not fully reach shoulder height, indicating a limitation in the range of motion.

FINAL_SCORE: 1


In [71]:
# 8
context_8 = \
"""
{
  "question": "How well does the patient perform forearm supination?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "instructions": "Task cue: 'Starting with your arm in your lap, bring your hand to your ear like you're answering the telephone. Get your elbow up to shoulder height, get your shoulder back, and flip your hand so your thumb is pointed toward the ceiling.' For this item, look only at the final position. Check whether the palm ends up facing upward without excessive shoulder or trunk compensation.",
  "scoring_criteria": {
    "0": "Cannot be performed. No forearm supination occurs.",
    "1": "Performed partially. Some forearm supination occurs.",
    "2": "Performed faultlessly. The back of the hand is facing forward, with more than half of it visible."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_8))

Unused or unrecognized kwargs: return_tensors, fps.


C00011/C00011_FM3_8_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Unused or unrecognized kwargs: return_tensors, fps.


The patient performs forearm supination by bringing her hand to her ear as if answering the telephone, raising her elbow to shoulder height, and then flipping her hand so that her thumb points toward the ceiling. Her palm ends up facing upward without any noticeable excessive shoulder or trunk compensation.

FINAL_SCORE: 2
S0001/S0001_FM3_8_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient brings his hand to his ear as if answering the telephone, raises his elbow to shoulder height, and then flips his hand so that his thumb points toward the ceiling. The palm appears to face upward, indicating successful forearm supination.

FINAL_SCORE: 2
S00021/S00021_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's forearm supination appears to be performed partially. While the hand is brought to the ear as if answering the telephone, the elbow reaches shoulder height, and the shoulder is slightly back, the palm does not fully face upward. There seems to be some compensatory movement from the shoulder rather than a clear supination.

FINAL_SCORE: 1
S0005/S0005_FM3_8_A_S.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's forearm supination appears to be performed partially. While some supination is observed, there seems to be noticeable shoulder or trunk compensation rather than a clean, faultless movement where the back of the hand faces forward.

FINAL_SCORE: 1


In [76]:
print(to_single_line_escaped(context_3))
print(to_single_line_escaped(context_4))
print(to_single_line_escaped(context_5))
print(to_single_line_escaped(context_6))
print(to_single_line_escaped(context_7))
print(to_single_line_escaped(context_8))

"{ ""question"": ""How well does the patient raise their shoulder (shoulder elevation)?"", ""response_instructions"": ""After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>."", ""instructions"": ""Task cue: 'Starting with your arm in your lap, bring your hand to your ear like you're answering the telephone. Get your elbow up to shoulder height, get your shoulder back, and flip your hand so your thumb is pointed toward the ceiling.' For this item, look only at the final position. Check if the shoulder itself is raised higher than the starting neutral position. Do not count extra leaning of the body or tilting of the head as correct. Base the score only on how the shoulder ends up."", ""scoring_criteria"": { ""0"": ""Cannot be performed. No shoulder lift at all, or needs help, or uses a lot of body/head lean instead."", ""1"": ""Performed partially. Shoulder lifts somewhat but not all the way, or depends a lot on leaning the body/head."", ""2"":

# 9-11

In [79]:
ds = load_strokerehab_ia_dataset(patients='all', fm_items='9-11', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']

In [ ]:
# 9
context_9 = \
"""
{
  "question": "How well does the patient perform shoulder adduction & internal rotation?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "instructions": "Task cue: 'Start from a position where your hand is next to your ear like you're answering the telephone. Fully straighten your elbow and reach across to touch your opposite knee, turning your palm down.' Check how well the patient gets their hand to the opposite knee at the end of the video. We allow assistance to attain the starting position, as well as arm/forearm support during the task.",
  "scoring_criteria": {
    "0": "Cannot be performed. Patient cannot move their arm/hand.",
    "1": "Performed partially. Patient can move their arm/hand but may require significant trunk lean or head tilt compensation.",
    "2": "Performed faultlessly. Patient can move their arm/hand to the opposite knee."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_9))

Unused or unrecognized kwargs: return_tensors, fps.


C00011/C00011_FM9_11_A_F.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be attempting shoulder adduction and internal rotation by reaching across to touch the opposite knee. However, she seems to struggle with the movement, possibly due to pain or difficulty in executing the full range of motion. She does not fully straighten her elbow and does not manage to bring her hand to the opposite knee.

FINAL_SCORE: 0
S0001/S0001_FM9_11_A_F.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be attempting to perform shoulder adduction and internal rotation by reaching across to touch their opposite knee. However, they seem to struggle with the movement, possibly due to pain or difficulty in executing the full range of motion. The patient's hand does not fully reach the opposite knee, indicating that the movement was not completed successfully.

FINAL_SCORE: 0
S00021/S00021_FM9_11_A_F.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be attempting to perform shoulder adduction and internal rotation by reaching across to touch the opposite knee. However, she seems to struggle significantly, requiring substantial assistance to even get her arm close to the starting position. Her movements are slow and labored, indicating limited mobility in her upper extremities.

FINAL_SCORE: 0
S0005/S0005_FM9_11_A_F.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be performing shoulder adduction and internal rotation by reaching across to touch the opposite knee. However, there seems to be some difficulty in fully straightening the elbow and reaching the opposite knee without requiring significant trunk lean or head tilt compensation.

FINAL_SCORE: 1


In [101]:
# 10
context_10 = \
"""
{
  "question": "At the end of the video, how well does the patient have their elbow extended?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "instructions": "Task cue: 'Start from a position where your hand is next to your ear like you're answering the telephone. Fully straighten your elbow and reach across to touch your opposite knee, turning your palm down.' Check how well the patient gets their elbow extended (straight) at the end of the video. We allow assistance to attain the starting position, as well as arm/forearm support during the task.",
  "scoring_criteria": {
    "0": "Cannot be performed Patient cannot complete the task, or their elbow is barely extended.",
    "1": "Performed partially. Elbow can move but cannot reach full extension, or requires significant trunk lean or head tilt compensation.",
    "2": "Performed faultlessly. Elbow can reach full extension with minimal compensation."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_10))

Unused or unrecognized kwargs: return_tensors, fps.


C00011/C00011_FM9_11_A_F.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to have their elbow extended to a point where it can reach close to their opposite knee, though there seems to be some difficulty in achieving full extension. The patient's arm is mostly straight, but there is a slight bend at the elbow, indicating that they are not fully extending it.

FINAL_SCORE: 1
S0001/S0001_FM9_11_A_F.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's elbow appears to be extended to a point where it can reach close to the opposite knee, though there might be some slight difficulty in achieving full extension without any compensation. The patient seems to be able to maintain this position for a moment before slightly bending the elbow again.

FINAL_SCORE: 1
S00021/S00021_FM9_11_A_F.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be in a wheelchair and is attempting to extend their elbow by reaching across to touch their opposite knee. However, it seems that they are having difficulty fully extending their elbow, as they are leaning forward and using their trunk for support rather than achieving full extension.

FINAL_SCORE: 0
S0005/S0005_FM9_11_A_F.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to have fully extended their elbow by reaching across to touch their opposite knee, with no significant trunk lean or head tilt compensation. The movement seems smooth and controlled.

FINAL_SCORE: 2


In [102]:
# 11
context_11 = \
"""
{
  "question": "At the end of the video, how well does the patient pronate their forearm (palm down)?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "instructions": "Task cue: 'Start from a position where your hand is next to your ear like you're answering the telephone. Fully straighten your elbow and reach across to touch your opposite knee, turning your palm down.' Check whether the hand reaches the knee at the video's end. Say so if not. If so, see how well the patient turns their palm down. We allow assistance to attain the starting position, as well as arm/forearm support during the task.",
  "scoring_criteria": {
    "0": "Cannot be performed. Patient cannot get their hand to reach their knee.",
    "1": "Performed partially. Hand can reach the knee, but the back of the hand is not fully visible.",
    "2": "Performed faultlessly. Hand can reach the knee, and the back of the hand is visible."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_11))

Unused or unrecognized kwargs: return_tensors, fps.


C00011/C00011_FM9_11_A_F.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to have reached their knee, but the back of the hand is not fully visible, indicating that the hand is not fully pronated.

FINAL_SCORE: 1
S0001/S0001_FM9_11_A_F.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's hand reaches the knee, but the back of the hand is not fully visible.

FINAL_SCORE: 1
S00021/S00021_FM9_11_A_F.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is seated in a wheelchair and appears to be attempting to pronate their forearm by reaching across to touch their opposite knee. However, the patient's hand does not reach the knee, and the palm is not turned down. The patient seems to need assistance to reach the knee.

FINAL_SCORE: 0
S0005/S0005_FM9_11_A_F.mp4


Unused or unrecognized kwargs: return_tensors, fps.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient successfully reaches the knee with their hand, and the back of the hand is clearly visible.

FINAL_SCORE: 2


In [108]:
print(to_single_line_escaped(context_9))
print(to_single_line_escaped(context_10))
print(to_single_line_escaped(context_11))

"{ ""question"": ""How well does the patient perform shoulder adduction & internal rotation?"", ""response_instructions"": ""After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>."", ""instructions"": ""Task cue: 'Start from a position where your hand is next to your ear like you're answering the telephone. Fully straighten your elbow and reach across to touch your opposite knee, turning your palm down.' Check how well the patient gets their hand to the opposite knee at the end of the video. We allow assistance to attain the starting position, as well as arm/forearm support during the task."", ""scoring_criteria"": { ""0"": ""Cannot be performed. Patient cannot move their arm/hand."", ""1"": ""Performed partially. Patient can move their arm/hand but may require significant trunk lean or head tilt compensation."", ""2"": ""Performed faultlessly. Patient can move their arm/hand to the opposite knee."" } }"
"{ ""question"": ""At the end of the vid

# 12-17

In [28]:
# 12
ds = load_strokerehab_ia_dataset(patients='all', fm_items='12', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_12 = \
"""
{
  "question": "How well does the patient get the back of their hand to touch their back?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "instructions": "Task cue: 'Place the back of your hand to your lower back.' Based on the positioning of the patient's hand and/or elbow, infer where the back of the patient's hand ends up at the end of the video.",
  "scoring_criteria": {
    "0": "The hand does not pass the side hip.",
    "1": "The hand passes the side hip but cannot reach the central lower back.",
    "2": "The hand reaches the central lower back."
  }
}
"""

# 1, 1, 1, 1 from LLM

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_12))

Unused or unrecognized kwargs: fps, return_tensors.


C00011/C00011_FM12_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's hand is positioned behind their trunk, touching the lower back area. The hand appears to be making clear contact with the back, indicating that the hand has moved sufficiently behind the trunk.

FINAL_SCORE: 2
S0001/S0001_FM12_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's hand appears to be moving behind their trunk but does not make clear contact with the lower back. It seems to stop just past the side.

FINAL_SCORE: 1
S00021/S00021_FM12_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's hand appears to be positioned behind the trunk, touching the lower back area. The hand is clearly making contact with the back, indicating that the hand has moved behind the trunk and is now resting on the lower back.

FINAL_SCORE: 2
S0005/S0005_FM12_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's hand appears to be moving behind their trunk but does not make clear contact with the lower back. It seems to stop just past the side.

FINAL_SCORE: 1


In [7]:
# 13
ds = load_strokerehab_ia_dataset(patients='all', fm_items='13', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_13 = \
"""
{
  "question": "How well does the patient flex their shoulder to 90 degrees?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "instructions": "Task cue: 'Start with your arm straight down at your side with your thumb pointing forward. Keep your elbow straight as you raise your arm up to shoulder height in front of you.' We allow assistance to attain the starting position.",
  "scoring_criteria": {
    "0": "Cannot be performed. Patient cannot maintain the starting position, or deviates at the start. Deviations include a bend at the elbow, shoulder abduction, or forearm pronation.",
    "1": "Performed partially. Patient can raise their arm but not to 90 degrees, or deviates after starting. Deviations include a bend at the elbow, shoulder abduction, or forearm pronation.",
    "2": "Performed faultlessly. Patient can raise their arm to 90 degrees without any deviations."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_13))

C00011/C00011_FM13_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is sitting on a chair and appears to be following the instructions given. She starts with her arm straight down at her side with her thumb pointing forward. She then raises her arm up to shoulder height in front of her, keeping her elbow straight. The patient successfully reaches shoulder height without any deviation, indicating that she has flexed her shoulder to 90 degrees.

FINAL_SCORE: 2
S0001/S0001_FM13_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is sitting on a chair and appears to be following instructions to flex their shoulder to 90 degrees. The therapist is assisting by holding the patient's arm in place. The patient raises their arm up to shoulder height in front of them, maintaining a straight elbow and keeping the thumb pointing forward. There are no visible deviations such as bending at the elbow, shoulder abduction, or forearm pronation.

FINAL_SCORE: 2
S00021/S00021_FM13_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is seated in a wheelchair and appears to be attempting to flex their shoulder to 90 degrees. The patient's arm is raised, but it seems to be deviating from the correct position, possibly bending at the elbow rather than maintaining a straight arm. The patient also appears to be having difficulty keeping the arm straight.

FINAL_SCORE: 0
S0005/S0005_FM13_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is sitting on a chair and appears to be attempting to raise their arm to shoulder height. However, they seem to have difficulty maintaining the starting position with their arm straight down at their side. The patient's arm bends at the elbow before reaching shoulder height.

FINAL_SCORE: 0


In [29]:
# 14
ds = load_strokerehab_ia_dataset(patients='all', fm_items='14', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_14 = \
"""
{
  "question": "How well does the patient perform forearm pronation-supination?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Tuck your arm in at your side in the shape of an 'L.' Without moving the rest of your arm, turn your palm up and down as far as you can go."
  "instructions": "First, comment on possible deviations from the expectation: the arm is in the 'L' shape, the shoulder is neutral during movement, the elbow is tucked in at trunk. Second, comment on the range of motion. Finally, give your final answer.",
  "scoring_criteria": {
    "0": "Cannot perform the movement, or one of the above deviations mentioned occurs at onset.",
    "1": "Can partially perform the movement, or one of the above deviations mentioned occurs later in movement.",
    "2": "No deviations, and good range of motion."
  }
}
"""

# LLM 1-1-1-1

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_14))

Unused or unrecognized kwargs: fps, return_tensors.


C00011/C00011_FM14_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's arm is in the 'L' shape, the shoulder appears to be neutral during movement, and the elbow is tucked in at the trunk. The patient can perform forearm pronation-supination, but there seems to be some difficulty in reaching full range of motion. The patient's movements appear somewhat limited.

FINAL_SCORE: 1
S0001/S0001_FM14_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's arm is in the 'L' shape, the shoulder appears to be neutral, and the elbow is tucked in at the trunk. The patient can partially perform the forearm pronation-supination, but there seems to be some difficulty in reaching full range of motion.

FINAL_SCORE: 1
S00021/S00021_FM14_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's arm is in the 'L' shape, the shoulder is neutral, and the elbow is tucked in at the trunk. The patient can turn their palm up and down, but there appears to be some difficulty in fully pronating and supinating the forearm. The range of motion seems limited.

FINAL_SCORE: 1
S0005/S0005_FM14_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's arm is in the 'L' shape, the shoulder is neutral, and the elbow is tucked in at the trunk. The patient can perform forearm pronation-supination to some extent, but there appears to be a slight deviation where the wrist may not fully extend to the neutral position. The range of motion seems limited compared to what would be considered normal.

FINAL_SCORE: 1


In [ ]:
# 15
ds = load_strokerehab_ia_dataset(patients='all', fm_items='15', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_15 = \
"""
{
  "question": "How well does the patient abduct their shoulder to 90 degrees?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Hold you arm straight out at your side and point your thumb forward. Keep your elbow straight as you raise your arm to the side up to shoulder height. Keep your palm facing down.",
  "instructions": "Comment on possible deviations from the expectation: the elbow is straight, the palm faces down. Also comment on the shoulder's height at the end. We allow assistance to the starting position.",
  "scoring_criteria": {
    "0": "Cannot be performed, or cannot attain the starting position, or deviates at onset.",
    "1": "Performed partially, or movement deviates later in movement, or range of motion is partial.",
    "2": "Performed faultlessly without deviations."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_15))

Unused or unrecognized kwargs: fps, return_tensors.


C00011/C00011_FM15_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient successfully raises their arm to shoulder height with the elbow straight and the palm facing down. The shoulder reaches the desired height without any noticeable deviations.

FINAL_SCORE: 2
S0001/S0001_FM15_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to have difficulty raising their arm to shoulder height while keeping the elbow straight and the palm facing down. The shoulder does not reach the desired height, indicating a limitation in abduction.

FINAL_SCORE: 0
S00021/S00021_FM15_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to raise their arm to shoulder height with the elbow straight and the palm facing down. However, there seems to be some difficulty in maintaining the abduction at 90 degrees, as the shoulder does not fully reach the desired height.

FINAL_SCORE: 1
S0005/S0005_FM15_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to have difficulty raising their arm to shoulder height while keeping the elbow straight and the palm facing down. The shoulder does not reach the desired height, indicating a limitation in abduction.

FINAL_SCORE: 0


In [30]:
# 16
ds = load_strokerehab_ia_dataset(patients='all', fm_items='16', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_16 = \
"""
{
  "question": "How well does the patient flex their shoulder from 90 degrees to 180 degrees?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Hold your arm straight out in front of your body with your thumb pointing up to the ceiling. Keep your elbow straight as you raise your arm up as far as you can.",
  "instructions": "Check that the shoulder flexes fully overhead (up to 180 degrees), the elbow remains straight, and the forearm stays neutral. Assistance to starting position is allowed. Comment on whether the final position is achieved and whether deviations appear during the motion.",
  "scoring_criteria": {
    "0": "Cannot be performed, or cannot attain the starting position, or deviates at onset.",
    "1": "Performed partially, or movement deviates later in movement, or range of motion is partial.",
    "2": "Performed faultlessly without deviations."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_16))

Unused or unrecognized kwargs: fps, return_tensors.


C00011/C00011_FM16_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient successfully raises her arm to 180 degrees, maintaining a straight elbow and a neutral forearm throughout the movement. There are no deviations noted during the motion.

FINAL_SCORE: 2
S0001/S0001_FM16_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to flex their shoulder from approximately 90 degrees to just over 180 degrees. The elbow remains relatively straight throughout the movement, and the forearm stays in a neutral position. There are no visible deviations during the motion.

FINAL_SCORE: 2
S00021/S00021_FM16_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to raise their arm up to approximately 180 degrees, but there seems to be some slight deviation in the movement, particularly towards the end where the shoulder may not fully extend to the desired 180 degrees. The elbow remains relatively straight throughout the motion, and the forearm stays neutral.

FINAL_SCORE: 1
S0005/S0005_FM16_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to extend their arm fully overhead to approximately 180 degrees, maintaining a straight elbow and a neutral forearm throughout the motion. There are no visible deviations noted during the movement.

FINAL_SCORE: 2


In [31]:
# 17
ds = load_strokerehab_ia_dataset(patients='all', fm_items='17', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_17 = \
"""
{
  "question": "How well does the patient perform forearm pronation supination?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Hold your arm straight in front but pointing down to the wall. Keeping your elbow straight, turn your palm up and down as far as you can.",
  "instructions": "Limit shoulder rotation compensation. The shoulder should stay flexed between 30 degrees and 90 degrees, the elbow straight, and the forearm should rotate through full pronation and supination comparable to the unaffected side. Note any reduced range, inconsistency, or compensatory motion.",
  "scoring_criteria": {
    "0": "Cannot be performed, or cannot attain the starting position, or deviates at onset.",
    "1": "Performed partially, or movement deviates later in movement, or range of motion is partial.",
    "2": "Performed faultlessly without deviations."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_17))

C00011/C00011_FM17_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be performing forearm pronation and supination with some difficulty. She is able to move her forearm through a range that seems comparable to the unaffected side, but there is noticeable inconsistency in the movement. Her shoulder remains relatively stable within the specified range, and she maintains an elbow that is close to being straight.

FINAL_SCORE: 1
S0001/S0001_FM17_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be performing forearm pronation and supination with some difficulty. While they are attempting to move their forearm up and down, there seems to be a slight deviation from the expected range of motion. The patient's shoulder remains relatively stable, but the movement of the forearm does not seem to reach the full extent compared to what would be considered normal for an unaffected side.

FINAL_SCORE: 1
S00021/S00021_FM17_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to have difficulty performing forearm pronation and supination. The shoulder seems to be compensating for the movement, and the range of motion is limited compared to what would be expected from an unaffected side.

FINAL_SCORE: 0
S0005/S0005_FM17_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient performs forearm pronation and supination with some difficulty. While he attempts to move his forearm up and down, there appears to be a slight deviation from the expected range of motion, possibly due to limited shoulder flexibility or muscle weakness. However, he manages to complete the movement without significant shoulder compensation.

FINAL_SCORE: 1


In [33]:
print(to_single_line_escaped(context_12))
print(to_single_line_escaped(context_13))
print(to_single_line_escaped(context_14))
print(to_single_line_escaped(context_15))
print(to_single_line_escaped(context_16))
print(to_single_line_escaped(context_17))

"{ ""question"": ""How well does the patient place the back of their hand behind their trunk toward the lower back?"", ""response_instructions"": ""After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>."", ""instructions"": ""Task cue: 'Bring the back of your hand behind you and touch your lower back, as if slipping it into your waistband at the back.' At the end of the motion, check if the hand is placed behind the trunk or only at the side/front. When in doubt, give the higher score."", ""scoring_criteria"": { ""0"": ""The hand never moves behind the trunk (stays at the side hip or in front)."", ""1"": ""The hand moves somewhat behind the trunk but does not touch the lower back (stops just past the side)."", ""2"": ""The hand is anywhere on the lower back, whether central or off to the side. If the hand makes clear contact with the back, award a 2."" } }"
"{ ""question"": ""How well does the patient flex their shoulder to 90 degrees?"", ""res

# 19-23

In [45]:
# 19
ds = load_strokerehab_ia_dataset(patients='all', fm_items='19', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_19 = \
"""
{
  "question": "How well does the patient maintain wrist stability at 90 degrees elbow flexion?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Tuck your arm in at your side in the shape of an 'L with your palm facing down. Without moving the rest of your arm, hold your wrist up. If you can hold your wrist up, I will push the back of your hand slightly.",
  "instructions": "Check if the patient can achieve at least 15 degrees of wrist extension and hold it. Then check whether the patient can hold that position when the examiner pushes the back of their hand. Note if the patient cannot maintain starting position, deviates at onset, or loses stability when resistance is applied.",
  "scoring_criteria": {
    "0": "Cannot achieve 15 degrees wrist extension, or cannot maintain starting position, or deviates at onset of extension",
    "1": "Achieves extension but shows no resistance, or deviates after resistance is applied.",
    "2": "Maintains position faultlessly with resistance."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_19))

Unused or unrecognized kwargs: fps, return_tensors.


C00011/C00011_FM19_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Unused or unrecognized kwargs: fps, return_tensors.


The patient appears to be able to hold her wrist up at 90 degrees of elbow flexion without deviation. When the examiner pushes the back of her hand, she maintains her wrist position without losing stability.

FINAL_SCORE: 2
S0001/S0001_FM19_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to hold his wrist up at approximately 90 degrees of elbow flexion without deviation. However, the video quality makes it difficult to determine if he maintains this position under resistance. Given the lack of clear visual evidence of resistance application, we must rely on the initial observation.

FINAL_SCORE: 1
S00021/S00021_FM19_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to hold their wrist up at 90 degrees of elbow flexion without deviation. However, upon applying resistance by pushing the back of their hand, they show some instability in maintaining the position.

FINAL_SCORE: 1
S0005/S0005_FM19_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to hold their wrist up at 90 degrees of elbow flexion without deviation. However, upon applying resistance by pushing the back of their hand, they show some instability in maintaining the position.

FINAL_SCORE: 1


In [49]:
# 20
ds = load_strokerehab_ia_dataset(patients='all', fm_items='20', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_20 = \
"""
{
  "question": "How good is the patient's wrist range of motion at 90 degrees elbow flexion?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Tuck your arm in at your side in the shape of an L with your palm facing down. Without moving the rest of your arm, move your wrist all the way up and down.",
  "instructions": "First, check whether the wrist moves at all from the starting position. If there is no visible motion, this is scored as 0. If there is some motion but the wrist cannot complete the full up and down cycle, or movement is irregular, this is scored as 1. Only if the wrist clearly moves through a full flexion–extension cycle smoothly and without deviations should it be scored as 2. Speed is not scored.",
  "scoring_criteria": {
    "0": "No visible wrist movement, or cannot maintain starting position, or deviates immediately at onset.",
    "1": "Some wrist movement occurs but it is incomplete, limited, or inconsistent, or deviations appear later in the motion.",
    "2": "Full flexion and extension cycle is completed smoothly without deviations."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_20))

Unused or unrecognized kwargs: fps, return_tensors.


C00011/C00011_FM20_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's wrist moves slightly but does not complete a full flexion-extension cycle smoothly. The movement appears to be limited and inconsistent.

FINAL_SCORE: 1
S0001/S0001_FM20_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's wrist moves slightly but does not complete a full flexion-extension cycle smoothly. The movement appears to be limited and inconsistent.

FINAL_SCORE: 1
S00021/S00021_FM20_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's wrist moves through a full flexion-extension cycle smoothly without any deviations.

FINAL_SCORE: 2
S0005/S0005_FM20_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's wrist moves up and down smoothly, completing a full flexion-extension cycle without any deviations. The movement appears consistent throughout the range.

FINAL_SCORE: 2


In [48]:
# 21
ds = load_strokerehab_ia_dataset(patients='all', fm_items='21', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_21 = \
"""
{
  "question": "How well does the patient maintain wrist stability with the elbow straight at 0 degrees?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Hold your arm straight in front but pointing down to the bottom of the wall. Keeping your elbow straight, hold your wrist up. If you can hold your wrist up, I will push the back of your hand slightly.",
  "instructions": "Check if the patient can achieve at least 15 degrees of wrist extension and hold it with the elbow at 0 degrees. Then check whether the patient can maintain that position when the examiner pushes the back of their hand. Note if the patient cannot maintain the starting position, deviates at onset, or loses stability when resistance is applied. Slight shoulder abduction is acceptable.",
  "scoring_criteria": {
    "0": "Cannot achieve 15 degrees wrist extension, or cannot maintain starting position, or deviates at onset of extension.",
    "1": "Achieves extension but shows no resistance, or deviates after resistance is applied.",
    "2": "Maintains position faultlessly with resistance."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_21))

Unused or unrecognized kwargs: fps, return_tensors.


C00011/C00011_FM21_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to hold her wrist up with the elbow straight at 0 degrees. However, upon applying resistance by pushing the back of her hand, she shows some deviation from the starting position. This indicates that she may not be able to maintain the position faultlessly under resistance.

FINAL_SCORE: 1
S0001/S0001_FM21_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to hold his wrist up with the elbow straight at 0 degrees. However, upon applying slight resistance by pushing the back of his hand, he seems to lose some stability and his wrist begins to deviate downward.

FINAL_SCORE: 1
S00021/S00021_FM21_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to hold their wrist up with the elbow straight at 0 degrees. However, upon applying slight resistance by pushing the back of their hand, they show some deviation from the starting position.

FINAL_SCORE: 1
S0005/S0005_FM21_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be maintaining wrist stability with the elbow straight at 0 degrees. The patient's wrist is held up, and there is no visible deviation or loss of stability when the examiner pushes the back of the patient's hand.

FINAL_SCORE: 2


In [50]:
# 22
ds = load_strokerehab_ia_dataset(patients='all', fm_items='22', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_22 = \
"""
{
  "question": "How good is the patient's wrist range of motion with the elbow straight at 0 degrees?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Hold your arm straight in front but pointing down to the bottom of the wall. Keeping your elbow straight, move your wrist all the way up and down.",
  "instructions": "First, check whether the wrist moves at all from the starting position. If there is no visible motion, this is scored as 0. If there is some motion but the wrist cannot complete the full up and down cycle, or movement is irregular, this is scored as 1. Only if the wrist clearly moves through a full flexion–extension cycle smoothly and without deviations should it be scored as 2. Speed is not scored.",
  "scoring_criteria": {
    "0": "No visible wrist movement, or cannot maintain starting position, or deviates immediately at onset.",
    "1": "Some wrist movement occurs but it is incomplete, limited, or inconsistent, or deviations appear later in the motion.",
    "2": "Full flexion and extension cycle is completed smoothly without deviations."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_22))

Unused or unrecognized kwargs: fps, return_tensors.


C00011/C00011_FM22_A_S.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's wrist moves up and down smoothly, completing a full flexion-extension cycle without any deviations. The movement appears consistent throughout the range.

FINAL_SCORE: 2
S0001/S0001_FM22_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's wrist moves slightly but does not complete a full flexion-extension cycle smoothly. There are some deviations in the motion, indicating that the wrist does not move through a full range of motion.

FINAL_SCORE: 1
S00021/S00021_FM22_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's wrist moves slightly but does not complete a full flexion-extension cycle smoothly. There are some deviations in the motion.

FINAL_SCORE: 1
S0005/S0005_FM22_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient's wrist moves up and down, but it does not complete a full flexion-extension cycle smoothly. There are some deviations in the motion, indicating that the wrist does not move through the full range of motion without irregularities.

FINAL_SCORE: 1


In [55]:
# 23
ds = load_strokerehab_ia_dataset(patients='all', fm_items='23', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_23 = \
"""
{
  "question": "How good is the patient's wrist circumduction with the elbow straight at 0 degrees?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Hold your arm straight in front but pointing down to the bottom of the wall. Keeping your elbow straight, rotate your wrist to draw a quarter-sized circle in space.",
  "instructions": "Judge both wrist motion and posture. The elbow must stay straight at 0 degrees, the shoulder flexed between 30-90 degrees, and the forearm pronated. Slight shoulder abduction is acceptable, but trunk lean, elbow bend, or large shoulder motion count as deviations. Start by checking if the patient maintains the starting position and initiates circular wrist movement. If they cannot maintain position or no circle is visible, score 0. If the circle is small, incomplete, or irregular but some attempt is made with posture mostly intact, score 1. Only score 2 if a full smooth circle is drawn and posture is maintained with at most minor acceptable deviations.",
  "scoring_criteria": {
    "0": "No visible wrist circumduction, OR cannot maintain starting position at onset, OR loses posture immediately (e.g., elbow bend, trunk lean, large shoulder motion).",
    "1": "Circumduction is attempted with some wrist movement but is incomplete, small, or irregular, OR posture deviations occur later in the movement.",
    "2": "Circumduction is complete and smooth, and the starting position is maintained with only minor acceptable deviations (e.g., slight shoulder abduction)."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_23))

C00011/C00011_FM23_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be attempting to perform wrist circumduction with the elbow straight at 0 degrees. However, the movement seems to be incomplete and irregular, and there are noticeable posture deviations such as slight shoulder abduction and trunk lean. Given these observations, the patient does not meet the criteria for a full smooth circle and maintaining the starting position with only minor acceptable deviations.

FINAL_SCORE: 1
S0001/S0001_FM23_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be attempting to perform wrist circumduction with their elbow straight at 0 degrees. However, the movement seems incomplete and irregular, and there may be some posture deviations such as slight shoulder abduction. Given these observations, the score would likely be:

FINAL_SCORE: 1
S00021/S00021_FM23_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be attempting to perform wrist circumduction with the elbow straight at 0 degrees. However, the movement seems incomplete and irregular, and there are noticeable posture deviations such as slight shoulder abduction and trunk lean. Given these observations, the patient does not meet the criteria for a full smooth circle and maintaining the starting position with only minor acceptable deviations.

FINAL_SCORE: 1
S0005/S0005_FM23_A_S.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be attempting to perform wrist circumduction with the elbow straight at 0 degrees. However, the movement seems to be incomplete and irregular, and there may be some posture deviations. Given these observations, I would score this attempt as 1.

FINAL_SCORE: 1


In [54]:
print(to_single_line_escaped(context_19))
print(to_single_line_escaped(context_20))
print(to_single_line_escaped(context_21))
print(to_single_line_escaped(context_22))
print(to_single_line_escaped(context_23))

"{ ""question"": ""How well does the patient maintain wrist stability at 90 degrees elbow flexion?"", ""response_instructions"": ""After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>."", ""task_cue"": ""Tuck your arm in at your side in the shape of an 'L with your palm facing down. Without moving the rest of your arm, hold your wrist up. If you can hold your wrist up, I will push the back of your hand slightly."", ""instructions"": ""Check if the patient can achieve at least 15 degrees of wrist extension and hold it. Then check whether the patient can hold that position when the examiner pushes the back of their hand. Note if the patient cannot maintain starting position, deviates at onset, or loses stability when resistance is applied."", ""scoring_criteria"": { ""0"": ""Cannot achieve 15 degrees wrist extension, or cannot maintain starting position, or deviates at onset of extension"", ""1"": ""Achieves extension but shows no resistance, or

In [64]:
ds = load_strokerehab_ia_dataset(patients='all', fm_items='24-25', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
print(ds['path_v'])


['C00011/C00011_FM24_25_A_F.mp4', 'S0001/S0001_FM24_25_A_F.mp4', 'S00021/S00021_FM24_25_A_F.mp4', 'S0005/S0005_FM24_25_A_F.mp4']


In [11]:
# 24
ds = load_strokerehab_ia_dataset(patients='all', fm_items='24', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_24 = \
"""
{
  "question": "Rate the patient's finger mass flexion.",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Make a tight fist and then open it with your fingers stretched as much as you can.",
  "instructions": "Comment on the patient's active finger movement and whether the patient can actively squeeze their fingers into a ball.",
  "scoring_criteria": {
    "0": "No finger flexion, or assistance required to achieve flexion.",
    "1": "Partial finger flexion.",
    "2": "Full finger flexion."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_24))

Unused or unrecognized kwargs: fps, return_tensors.


C00011/C00011_FM24_25_A_F.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is attempting to make a tight fist but appears to be struggling slightly. The fingers are not fully extended when the patient opens them. There seems to be some difficulty in achieving full flexion.

FINAL_SCORE: 1
S0001/S0001_FM24_25_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is attempting to make a tight fist but struggles to fully extend his fingers. He appears to be able to partially flex his fingers into a ball, but they do not reach full extension.

FINAL_SCORE: 1
S00021/S00021_FM24_25_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is attempting to make a tight fist but struggles to fully extend the fingers. The patient appears to be able to partially flex the fingers into a ball, but there is noticeable difficulty in achieving full extension.

FINAL_SCORE: 1
S0005/S0005_FM24_25_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is attempting to make a tight fist but struggles to fully extend the fingers. The patient's fingers do not reach a full extension when they try to open them after making a fist.

FINAL_SCORE: 1


In [12]:
# 25
ds = load_strokerehab_ia_dataset(patients='all', fm_items='25', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_25 = \
"""
{
  "question": "Rate the patient's finger mass extension.",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Make a tight fist and then open it with your fingers stretched as much as you can.",
  "instructions": "Comment on the patient's active finger movement and whether the patient can actively extend their fingers to be fully straight.",
  "scoring_criteria": {
    "0": "No finger extension, or assistance required to achieve extension.",
    "1": "Partial finger extension.",
    "2": "Full finger extension."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_25))

Unused or unrecognized kwargs: fps, return_tensors.


C00011/C00011_FM24_25_A_F.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is attempting to extend her fingers but they do not reach full extension. The patient appears to need some assistance to achieve full finger extension.

FINAL_SCORE: 1
S0001/S0001_FM24_25_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is attempting to extend his fingers but appears to have difficulty achieving full extension. His fingers are partially extended, indicating some degree of movement but not complete straightness.

FINAL_SCORE: 1
S00021/S00021_FM24_25_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is actively extending their fingers, but they do not reach full straightness. The patient appears to have some difficulty in achieving complete finger extension without assistance.

FINAL_SCORE: 1
S0005/S0005_FM24_25_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is attempting to extend their fingers but appears to have difficulty achieving full extension. The fingers are not fully straight, indicating partial extension.

FINAL_SCORE: 1


In [13]:
print(to_single_line_escaped(context_24))
print(to_single_line_escaped(context_25))

"{ ""question"": ""Rate the patient's finger mass flexion."", ""response_instructions"": ""After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>."", ""task_cue"": ""Make a tight fist and then open it with your fingers stretched as much as you can."", ""instructions"": ""Comment on the patient's active finger movement and whether the patient can actively squeeze their fingers into a ball."", ""scoring_criteria"": { ""0"": ""No finger flexion, or assistance required to achieve flexion."", ""1"": ""Partial finger flexion."", ""2"": ""Full finger flexion."" } }"
"{ ""question"": ""Rate the patient's finger mass extension."", ""response_instructions"": ""After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>."", ""task_cue"": ""Make a tight fist and then open it with your fingers stretched as much as you can."", ""instructions"": ""Comment on the patient's active finger movement and whether the patient can ac

# 26-30

In [32]:
# 26
ds = load_strokerehab_ia_dataset(patients='all', fm_items='26', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_26 = \
"""
{
  "question": "Rate the patient's ability to perform Hook Grasp.",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Make your hand look like a hook or claw; don't let me pull your fingers straight.",
  "instructions": "Support at arm/forearm is permitted during the task. The patient should flex the distal part of the fingers (>= 45 degrees) while keeping the proximal part (at MCP) extended to 0 degrees. The examiner applies resistance by trying to pull the patient's fingers into extension. Judge the patient's ability to attain and hold the hook-like position against resistance.",
  "scoring_criteria": {
    "0": "Cannot attain the starting hook position, even with support.",
    "1": "Attains the hook-like position but cannot withstand resistance; the grip gives way.",
    "2": "Holds the hook grasp position firmly against resistance."
  }
}
"""


In [33]:
for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_26))

Unused or unrecognized kwargs: fps, return_tensors.


C00011/C00011_FM26_A_F.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to attain the hook-like position, but the grip does not seem to withstand the examiner's resistance. The patient's fingers flex slightly but do not maintain the required 45-degree flexion against the examiner's pull.

FINAL_SCORE: 1
S0001/S0001_FM26_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to attain the hook-like position, as indicated by the flexion of the distal part of the fingers and the extension of the proximal part. However, the patient seems to struggle slightly when the examiner tries to pull the fingers into extension, indicating that the grip does not hold firm against resistance.

FINAL_SCORE: 1
S00021/S00021_FM26_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is attempting to perform a hook grasp by flexing their distal fingers while keeping the proximal part extended. However, the examiner is applying resistance which causes the patient's fingers to give way. The patient is able to attain the hook-like position but cannot maintain it against the examiner's resistance.

FINAL_SCORE: 1
S0005/S0005_FM26_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is able to attain the hook-like position but struggles to maintain it against the examiner's resistance. The patient's fingers flex to approximately 45 degrees, but the grip does not hold steady.

FINAL_SCORE: 1


In [34]:
# 27
ds = load_strokerehab_ia_dataset(patients='all', fm_items='27', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_27 = \
"""
{
  "question": "Rate the patient's ability to perform Thumb Adduction.",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Hold this paper between your thumb and index finger, keeping them both straight. Don't let me pull it away.",
  "instructions": "Support at arm/forearm is permitted during the task. Starting position: fingers extended to 0 degrees, thumb abducted to admit paper. The paper is placed between the side of the thumb and index finger. The examiner applies a horizontal tug away from the subject. Limit thumb flexion and adduction to only what is required; knuckles may bend while resisting. If the thumb flexes during the tug, repeat to ensure the subject understands the task.",
  "scoring_criteria": {
    "0": "Cannot grasp the paper OR cannot attain the starting position OR thumb flexes immediately with tug.",
    "1": "Can hold the paper initially but cannot maintain grip against the tug.",
    "2": "Object is held firmly against the tug without thumb flexion or loss of grasp."
  }
}
"""

In [35]:
for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_27))

C00011/C00011_FM27_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Unused or unrecognized kwargs: fps, return_tensors.


The patient appears to be able to hold the paper between their thumb and index finger, maintaining a straight position for the initial moment. However, as the examiner applies a horizontal tug, the patient's thumb begins to flex slightly, indicating that they are not able to maintain the starting position without thumb flexion.

FINAL_SCORE: 0
S0001/S0001_FM27_A_F.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to hold the paper between their thumb and index finger, maintaining a straight position for the thumb. However, as the examiner applies a horizontal tug, the patient's thumb begins to flex slightly, indicating that they are not able to maintain the initial position without thumb flexion.

FINAL_SCORE: 1
S00021/S00021_FM27_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is able to hold the paper between their thumb and index finger, maintaining the starting position with the thumb abducted. However, as the examiner applies a horizontal tug, the patient's thumb begins to flex slightly, indicating that they are not able to maintain the initial abduction position against the tug.

FINAL_SCORE: 1
S0005/S0005_FM27_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is holding the paper between their thumb and index finger, maintaining a straight position for both digits. They appear to be able to resist the examiner's tug without allowing the thumb to flex or lose grip on the paper. The patient seems to understand the task and is performing it correctly.

FINAL_SCORE: 2


In [36]:
# 28
ds = load_strokerehab_ia_dataset(patients='all', fm_items='28', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_28 = \
"""
{
  "question": "Rate the patient's ability to perform Pincer Grasp.",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Take the pencil from me with the pads of your thumb and index finger. Don’t let me pull it away.",
  "instructions": "Support at arm/forearm is permitted during the task. Starting position: thumb and index finger extended enough to fit a pencil between pads. Present the pencil vertically. Ensure it is held with both distal pads of thumb and index finger (not on the proximal portions). The examiner applies an upward tug to the pencil.",
  "scoring_criteria": {
    "0": "Cannot grasp the pencil OR cannot attain the starting position.",
    "1": "Can hold the pencil initially but cannot maintain grip against the tug.",
    "2": "Pencil is held firmly against the tug using thumb and index finger pads."
  }
}
"""

In [37]:
for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_28))

C00011/C00011_FM28_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Unused or unrecognized kwargs: fps, return_tensors.


The patient successfully grasps the pencil with the pads of her thumb and index finger and maintains a firm grip even when the examiner applies an upward tug. This indicates that she can perform the Pincer Grasp effectively.

FINAL_SCORE: 2
S0001/S0001_FM28_A_F.mp4


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient successfully holds the pencil with the pads of their thumb and index finger, maintaining a firm grip even as the examiner applies an upward tug. This indicates that the patient can perform the Pincer Grasp effectively.

FINAL_SCORE: 2
S00021/S00021_FM28_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is able to hold the pencil with the pads of their thumb and index finger, but they struggle to maintain a firm grip against the examiner's upward tug. They appear to be using some support from their forearm, which is not allowed according to the scoring criteria.

FINAL_SCORE: 1
S0005/S0005_FM28_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient successfully holds the pencil with the pads of their thumb and index finger, maintaining a firm grip even as the examiner applies an upward tug. This indicates that the patient can perform the Pincer Grasp effectively.

FINAL_SCORE: 2


In [38]:
# 29
ds = load_strokerehab_ia_dataset(patients='all', fm_items='29', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_29 = \
"""
{
  "question": "Rate the patient's ability to perform Cylindrical Grasp.",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Make an OK sign and take the jar from me with only your thumb and index finger wrapped around it. Don’t let me pull it away.",
  "instructions": "Support at arm/forearm is permitted during the task. Starting position: thumb and index finger extended enough to fit around a cylindrical object (e.g., spice jar). The patient should hold the cylinder at its base with thumb and index finger. The examiner applies an upward tug to the jar.",
  "scoring_criteria": {
    "0": "Cannot grasp the cylinder OR cannot attain the starting position.",
    "1": "Can hold the jar initially but cannot maintain grip against the tug.",
    "2": "Cylinder is held firmly against the tug with thumb and index finger."
  }
}
"""


In [39]:

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_29))

C00011/C00011_FM29_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient successfully holds the jar with her thumb and index finger, maintaining a firm grip even as the examiner applies an upward tug. She does not let go of the jar despite the examiner's attempt to pull it away.

FINAL_SCORE: 2
S0001/S0001_FM29_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is attempting to perform a cylindrical grasp by extending their thumb and index finger to fit around a cylindrical object. However, they seem to struggle to maintain a firm grip against the examiner's upward tug. The patient's hand movements indicate difficulty in holding the cylinder securely.

FINAL_SCORE: 1
S00021/S00021_FM29_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient successfully holds the jar with their thumb and index finger, maintaining a firm grip even as the examiner applies an upward tug. This indicates that the patient can perform the Cylindrical Grasp task effectively.

FINAL_SCORE: 2
S0005/S0005_FM29_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is able to hold the jar with his thumb and index finger, but he struggles to maintain his grip against the examiner's upward tug. He appears to be using some support from his forearm, which is not allowed according to the scoring criteria.

FINAL_SCORE: 1


In [40]:
# 30
ds = load_strokerehab_ia_dataset(patients='all', fm_items='30', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_30 = \
"""
{
  "question": "Rate the patient's ability to perform Spherical Grasp.",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "task_cue": "Grasp the tennis ball with the tips of all five fingers from my hand. Don't let me pull it out of your grasp.",
  "instructions": "Support at arm/forearm is permitted during the task. Starting position: fingers extended and abducted enough to fit around a tennis ball. Patient may allow the ball to touch the palm. The examiner gives the ball a tug in the opposite direction of the palm.",
  "scoring_criteria": {
    "0": "Cannot grasp the ball OR cannot attain the starting position.",
    "1": "Can hold the ball initially but cannot maintain grip against the tug.",
    "2": "Tennis ball is held firmly against the tug with thumb and fingers in abducted and flexed position."
  }
}
"""


In [41]:

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_30))

C00011/C00011_FM30_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient can initially hold the ball but struggles to maintain grip against the examiner's tug. The ball slips out of her grasp as she attempts to hold it.

FINAL_SCORE: 1
S0001/S0001_FM30_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be able to grasp the tennis ball with the tips of all five fingers, maintaining a firm hold even as the examiner tugs on it. The patient's thumb and fingers are abducted and flexed, indicating a good grasp.

FINAL_SCORE: 2
S00021/S00021_FM30_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient is able to grasp the tennis ball with the tips of all five fingers, maintaining a firm grip even when the examiner tugs on it. The patient's thumb and fingers are abducted and flexed correctly.

FINAL_SCORE: 2
S0005/S0005_FM30_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient can hold the ball initially but cannot maintain grip against the examiner's tug.

FINAL_SCORE: 1


In [29]:
print(to_single_line_escaped(context_26))
print(to_single_line_escaped(context_27))
print(to_single_line_escaped(context_28))
print(to_single_line_escaped(context_29))
print(to_single_line_escaped(context_30))

"{ ""question"": ""Rate the patient's ability to perform Hook Grasp."", ""response_instructions"": ""After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>."", ""task_cue"": ""Make your hand look like a hook or claw; don't let me pull your fingers straight."", ""instructions"": ""Support at arm/forearm is permitted during the task. The patient should flex the distal part of the fingers (>= 45 degrees) while keeping the proximal part (at MCP) extended to 0 degrees. The examiner applies resistance by trying to pull the patient's fingers into extension. Judge the patient's ability to attain and hold the hook-like position against resistance."", ""scoring_criteria"": { ""0"": ""Cannot attain the starting hook position, even with support."", ""1"": ""Attains the hook-like position but cannot withstand resistance; the grip gives way."", ""2"": ""Holds the hook grasp position firmly against resistance."" } }"
"{ ""question"": ""Rate the patient's abili

# 31-33

In [43]:
# 31
ds = load_strokerehab_ia_dataset(patients='all', fm_items='31', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_31 = \
"""
{
  "question": "Assess the patient's tremor (oscillations during movement).",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "instructions": "Observe the presence and impact of tremor during the patient's movement. Focus on whether the oscillations alter the trajectory or not. If no movement is possible, score as 2.",
  "scoring_criteria": {
    "0": "Tremor is present throughout movement and significantly affects trajectory.",
    "1": "Tremor is present throughout movement but does not significantly affect trajectory.",
    "2": "No tremor observed OR patient is unable to move."
  }
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_31))

C00011/C00011_FM31_33_H_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be performing an exercise involving arm movements. The patient's arms show slight tremors, which seem to slightly affect the smoothness of the movements. However, the patient manages to maintain some control over the movements despite the tremors.

FINAL_SCORE: 1
C00011/C00011_FM31_33_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be performing a task that involves hand movements. Throughout the observed movements, there seems to be a noticeable tremor in her hands, which slightly affects the precision of her actions. However, she manages to continue the task without significant disruption.

FINAL_SCORE: 1
S0001/S0001_FM31_33_A_F.mp4


...

The patient in the wheelchair appears to have some hand movements that could be interpreted as tremors. However, these movements do not seem to significantly alter the trajectory of her actions. She is able to point and gesture, which suggests that while there might be some tremor, it does not impede her ability to perform these tasks.

FINAL_SCORE: 1
S0005/S0005_FM31_33_H_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be performing a series of movements that involve arm flexion and extension. Throughout these movements, there is noticeable tremor in the arms, which slightly alters the trajectory of the limbs. The patient's movements are somewhat affected by this tremor, making it difficult to maintain a smooth path.

FINAL_SCORE: 1
S0005/S0005_FM31_33_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be performing a series of movements that involve arm and leg actions. Throughout these movements, there is noticeable tremor in the arms, which slightly alters the trajectory of the limbs. The patient's movements are somewhat affected by this tremor, making it difficult to maintain a steady path.

FINAL_SCORE: 1


In [44]:
# 32
ds = load_strokerehab_ia_dataset(patients='all', fm_items='32', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_32 = \
"""
{
  "question": "Assess the patient's dysmetria (error in endpoint, misses the endpoint).",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>.",
  "instructions": "Observe the accuracy of the patient's endpoint control during movement. Focus on whether the dysmetria is pronounced and random, slight and systematic, or absent. If no movement is possible, score as 2.",
  "scoring_criteria": {
    "0": "Pronounced and random dysmetria.",
    "1": "Slight and systematic dysmetria.",
    "2": "No dysmetria observed OR patient is unable to move."
  }
}

"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_32))

C00011/C00011_FM31_33_H_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be performing a task that involves reaching for objects. The movements seem to be somewhat uncoordinated, but there isn't a clear indication of a pronounced or random error in endpoint control. The movements appear to be slightly off-target, suggesting a slight and systematic dysmetria.

FINAL_SCORE: 1
C00011/C00011_FM31_33_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be performing a task that involves reaching out with her arm. Her movements seem to be somewhat erratic, suggesting an error in endpoint control. The patient's arm does not appear to reach the intended target consistently, indicating a possible dysmetria.

FINAL_SCORE: 1
S0001/S0001_FM31_33_A_F.mp4


...

The patient in the wheelchair appears to be making hand movements that seem slightly off-target, suggesting a slight and systematic error in endpoint control. The movements do not appear to be completely random but rather show a consistent pattern of missing the intended target.

FINAL_SCORE: 1
S0005/S0005_FM31_33_H_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be performing a movement that involves reaching out with both hands. The movements seem to be slightly off-target, indicating a slight and systematic error in endpoint control. This suggests a mild dysmetria.

FINAL_SCORE: 1
S0005/S0005_FM31_33_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient appears to be performing a movement that involves reaching out and then bringing their hand back towards themselves. The movement seems to be somewhat uncontrolled, with the hand not reaching the intended endpoint consistently. There is a noticeable error in the endpoint, where the hand either overshoots or undershoots the target area.

FINAL_SCORE: 1


In [45]:
# 33a
ds = load_strokerehab_ia_dataset(patients='all', fm_items='33', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_33a = \
"""
{
  "question": "How many times did the patient touch their knee?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <number>.",
  "instructions": "Count each distinct time the patient's hand makes contact with their knee during the task. Do not include near-misses or partial motions unless there is clear contact.",
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_33a))

C00011/C00011_FM31_33_H_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient touched their knee twice during the task.

FINAL_SCORE: 2
C00011/C00011_FM31_33_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient touched their knee twice during the task.

FINAL_SCORE: 2
S0001/S0001_FM31_33_A_F.mp4


...

The patient's hand touches their knee twice during the task.

FINAL_SCORE: 2
S0005/S0005_FM31_33_H_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient touched their knee four times during the task.

FINAL_SCORE: 4
S0005/S0005_FM31_33_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient touched their knee three times during the task.

FINAL_SCORE: 3


In [46]:
# 33b
ds = load_strokerehab_ia_dataset(patients='all', fm_items='33', metadata_path=DataPaths.IA_VIDEO_METADATA_PATH3)['test']
context_33b = \
"""
{
  "question": "How many times did the patient touch their nose?",
  "response_instructions": "After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <number>.",
  "instructions": "Count each distinct time the patient's hand makes contact with their nose during the task. Do not include near-misses or partial motions unless there is clear contact.",
}
"""

for i in range(len(ds)):
    path_v = ds['path_v'][i]
    print(path_v)
    print(get_output(path_v, context_33b))

C00011/C00011_FM31_33_H_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient touched their nose once during the task.

FINAL_SCORE: 1
C00011/C00011_FM31_33_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient touched their nose once during the task.

FINAL_SCORE: 1
S0001/S0001_FM31_33_A_F.mp4


...

The patient touched their nose twice during the task.

FINAL_SCORE: 2
S0005/S0005_FM31_33_H_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient touched their nose twice during the task.

FINAL_SCORE: 2
S0005/S0005_FM31_33_A_F.mp4


Unused or unrecognized kwargs: fps, return_tensors.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The patient touched their nose twice during the task.

FINAL_SCORE: 2


In [47]:
print(to_single_line_escaped(context_31))
print(to_single_line_escaped(context_32))
print(to_single_line_escaped(context_33a))
print(to_single_line_escaped(context_33b))

"{ ""question"": ""Assess the patient's tremor (oscillations during movement)."", ""response_instructions"": ""After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>."", ""instructions"": ""Observe the presence and impact of tremor during the patient's movement. Focus on whether the oscillations alter the trajectory or not. If no movement is possible, score as 2."", ""scoring_criteria"": { ""0"": ""Tremor is present throughout movement and significantly affects trajectory."", ""1"": ""Tremor is present throughout movement but does not significantly affect trajectory."", ""2"": ""No tremor observed OR patient is unable to move."" } }"
"{ ""question"": ""Assess the patient's dysmetria (error in endpoint, misses the endpoint)."", ""response_instructions"": ""After noting your observations, end your reply with a single line exactly: FINAL_SCORE: <0|1|2>."", ""instructions"": ""Observe the accuracy of the patient's endpoint control during movement. F